# 01 — Shared Preprocessing

**Module 1 — Preprocessing**

Turns raw OSM waterways + Sentinel-2 into a per-region riparian buffer and a per-pixel feature
table, for every region in `REGIONS` below. Output of this notebook is the input to
`02_modelling.ipynb` — the two must agree on file names, which is why every save call below
writes to `../data/processed/{region}_...` and nothing else touches that pattern.

**Regions:** Kasarani (Pamoja Trust's ground-truth case study — bounding box, matches the
original problem statement) plus two more river reaches along the same Nairobi river system —
**Gatharaini** and **Motoine** — added so the pipeline is checked against more than one place
before anyone claims it generalizes. Both are real named entries in the OSM waterways shapefile
(`Gatharaini River`, 12.98km; `Motoine River`, 13.17km) — not guessed coordinates.

In [1]:
import json

import ee
import geopandas as gpd
import pandas as pd
from shapely.geometry import box

ee.Initialize(project='solar-haven-349708')

WATERWAYS_PATH = '../data/vectors/gis_osm_waterways_free_1.shp'
OUT_DIR = '../data/processed'

## Step 1 — Define regions

**One consistent methodology for all three regions: a fixed-radius circle around a center
point** — not a hand-picked bounding box. This matters more than it looks: an earlier draft of
this pipeline used a bounding box for Kasarani (`36.80,-1.32` to `36.95,-1.20`) that turned out
to cover ~220km², nearly 8x larger than the ~28km² 3km-radius circle the project's original
calibration against Pamoja Trust's field count actually used. A same-named region with a
different extent is not the same region — it screens a different set of buildings and would
silently invalidate the calibration in `03_fusion_and_report.ipynb`. Using one radius
(`CASE_STUDY_RADIUS_KM`) for every region also keeps results comparable across them and keeps
each Earth Engine query small enough to run in seconds instead of minutes (a full-bbox query
for just Kasarani pulled over 500,000 buildings and took over a minute just to count).

- **Kasarani** — center matches the original validated case study exactly.
- **Gatharaini / Motoine** — center is the geometric midpoint of the named river's own
  geometry (no independent ground truth exists for either, so there's nothing else to match).

In [2]:
CASE_STUDY_RADIUS_KM = 3

REGIONS = {
    'Kasarani':   {'method': 'point', 'center': (36.8969, -1.2296)},
    'Gatharaini': {'method': 'river_name', 'river_name': 'Gatharaini River'},
    'Motoine':    {'method': 'river_name', 'river_name': 'Motoine River'},
}

## Step 2 — Load + clip waterways, build the riparian buffer

Shared skeleton agreed by the whole team's decks: reproject to a metric CRS (EPSG:32737 — UTM
37S, correct hemisphere for Nairobi) before buffering, since a buffer distance in degrees isn't
a reliable metre distance. Buffer, dissolve overlapping reaches, reproject back to EPSG:4326 so
it overlays cleanly on standard web maps and Sentinel-2 exports.

In [3]:
def load_waterways():
    return gpd.read_file(WATERWAYS_PATH)


def get_region_center(waterways, region_key):
    cfg = REGIONS[region_key]
    if cfg['method'] == 'point':
        return cfg['center']
    river = waterways[waterways['name'] == cfg['river_name']]
    if river.empty:
        raise ValueError(f"No waterway named {cfg['river_name']!r} found in {WATERWAYS_PATH}")
    midpoint_metric = river.to_crs(epsg=32737).union_all().centroid
    midpoint = gpd.GeoSeries([midpoint_metric], crs=32737).to_crs(epsg=4326).iloc[0]
    return (midpoint.x, midpoint.y)


def get_region_aoi(center, radius_km=CASE_STUDY_RADIUS_KM):
    """A fixed-radius circle, approximated in geopandas as its bounding box — the same shape a
    same-radius `ee.Geometry.Point(center).buffer(radius_km * 1000)` produces for the Earth
    Engine side below, kept a plain box here since it's just used for clipping vectors."""
    lon, lat = center
    pad_deg = radius_km / 111.0
    return box(lon - pad_deg, lat - pad_deg, lon + pad_deg, lat + pad_deg)


def build_riparian_buffer(waterways, aoi, buffer_m=60):
    """Clip to aoi, keep real flowing water (drop drains/canals), buffer + dissolve."""
    clipped = waterways[waterways.intersects(aoi)].copy()
    rivers_only = clipped[clipped['fclass'].isin(['river', 'stream'])].copy()
    if rivers_only.empty:
        raise ValueError('No river/stream features intersect this AOI — check the AOI bounds.')
    rivers_metric = rivers_only.to_crs(epsg=32737)
    buffer_metric = rivers_metric.buffer(buffer_m)
    dissolved = gpd.GeoSeries([buffer_metric.union_all()], crs=32737)
    buffer_global = dissolved.to_crs(epsg=4326)
    exploded = gpd.GeoDataFrame(geometry=buffer_global.explode(index_parts=False).reset_index(drop=True))
    return rivers_only, exploded

Also save the clipped river *lines* (not just the buffer polygon) in the metric CRS —
`03_fusion_and_report.ipynb` needs true distance-to-river per building, which a 60m buffer
polygon alone can't give (calibration later sweeps distances well below 60m).

In [4]:
def save_river_lines(rivers_only, region_key):
    rivers_metric = rivers_only.to_crs(epsg=32737)[['name', 'fclass', 'geometry']]
    rivers_metric.to_file(f'{OUT_DIR}/{region_key.lower()}_rivers.geojson', driver='GeoJSON')

## Step 3 — Sentinel-2 composite with the full band set

The team's existing `kasarani_sentinel_tile.tif` export only kept B4/B3/B2 (RGB) — that cannot
produce NDVI, NDBI, or NDWI, all of which Module 2 needs (NDWI and distance-to-river are new —
raised while comparing models, kept as an experiment for Module 2 to test with/without rather
than assumed to help). This pulls the full set (`B2,B3,B4,B8,B11,B12`) plus SCL for
cloud/shadow masking, proven in the original solo pipeline.

In [5]:
BANDS = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']


def _mask_s2_clouds(image):
    scl = image.select('SCL')
    mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return image.updateMask(mask)


def get_sentinel2_composite(aoi_ee, start_date, end_date, cloud_threshold=20):
    s2 = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(aoi_ee)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_threshold))
        .map(_mask_s2_clouds)
    )
    scene_count = s2.size().getInfo()
    composite = s2.select(BANDS + ['SCL']).median().clip(aoi_ee)
    return composite.select(BANDS), scene_count


def build_feature_image(composite):
    ndvi = composite.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndbi = composite.normalizedDifference(['B11', 'B8']).rename('NDBI')
    ndwi = composite.normalizedDifference(['B3', 'B8']).rename('NDWI')  # Green/NIR — water index
    return composite.addBands(ndvi).addBands(ndbi).addBands(ndwi)


def get_river_distance_image(rivers_only, aoi_ee, search_radius=2000, max_error=10):
    """Distance-to-river as its own band, computed from the same clipped OSM rivers used for the
    buffer above — not a separate/looser dataset. Added to the feature table as an experiment,
    not an assumption: distance-to-river is exactly the geography this project cares about, but
    it also risks teaching the classifier this training region's specific river layout rather
    than a general built-up signal. `02_modelling.ipynb` runs the comparison with and without it
    and reports which actually helps, instead of asserting either way."""
    river_geojson = json.loads(rivers_only.to_json())
    river_fc = ee.FeatureCollection(river_geojson)
    return river_fc.distance(searchRadius=search_radius, maxError=max_error).clip(aoi_ee).rename('dist_to_river_m')

## Step 4 — Feature table: label with real ground truth, not thresholds

Per an earlier team finding: use ESA WorldCover as the real ground-truth label
instead of an NDVI/NDBI threshold guess. Threshold-based pseudo-labels stay available as a
documented fallback only (`fallback_ndvi_builtup_mask`) for quick local testing without network
access to WorldCover — never for the table Module 2 actually trains on.

In [6]:
def get_worldcover_builtup(aoi_ee):
    worldcover = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map').clip(aoi_ee)
    return worldcover.eq(50).rename('builtup')


def fallback_ndvi_builtup_mask(composite, ndvi_threshold=0.3):
    """Fallback only — see module docstring. Not used in the sampled feature table below."""
    ndvi = composite.normalizedDifference(['B8', 'B4'])
    return ndvi.lt(ndvi_threshold).rename('builtup')


def sample_feature_table(feature_image, builtup_label, aoi_ee, num_points=800, seed=42):
    training_image = feature_image.addBands(builtup_label)
    samples = training_image.stratifiedSample(
        numPoints=num_points,
        classBand='builtup',
        region=aoi_ee,
        scale=10,
        seed=seed,
        geometries=True,
    )
    features = samples.getInfo()['features']
    rows = []
    for f in features:
        row = dict(f['properties'])
        lon, lat = f['geometry']['coordinates']
        row['lon'], row['lat'] = lon, lat
        rows.append(row)
    return pd.DataFrame(rows)

## Step 5 — Run all three regions, save outputs

Output file names Module 2 depends on: `{region}_riparian_buffer.geojson` and
`{region}_feature_table.csv`.

In [7]:
waterways = load_waterways()
summary = {}

for region_key in REGIONS:
    print(f'--- {region_key} ---')
    center = get_region_center(waterways, region_key)
    aoi = get_region_aoi(center)
    aoi_ee = ee.Geometry.Point(list(center)).buffer(CASE_STUDY_RADIUS_KM * 1000)

    rivers_only, buffer_gdf = build_riparian_buffer(waterways, aoi, buffer_m=60)
    buffer_path = f'{OUT_DIR}/{region_key.lower()}_riparian_buffer.geojson'
    buffer_gdf.to_file(buffer_path, driver='GeoJSON')
    save_river_lines(rivers_only, region_key)

    composite, scene_count = get_sentinel2_composite(aoi_ee, '2024-01-01', '2024-12-31')
    feature_image = build_feature_image(composite)
    dist_band = get_river_distance_image(rivers_only, aoi_ee)
    feature_image = feature_image.addBands(dist_band)
    builtup_label = get_worldcover_builtup(aoi_ee)
    feature_table = sample_feature_table(feature_image, builtup_label, aoi_ee)
    table_path = f'{OUT_DIR}/{region_key.lower()}_feature_table.csv'
    feature_table.to_csv(table_path, index=False)

    summary[region_key] = {
        'river_reaches_in_aoi': len(rivers_only),
        'buffer_polygons': len(buffer_gdf),
        's2_scene_count': scene_count,
        'feature_table_rows': len(feature_table),
        'feature_table_builtup_pct': round(100 * feature_table['builtup'].mean(), 1) if len(feature_table) else None,
    }
    print(summary[region_key])

pd.DataFrame(summary).T

--- Kasarani ---


{'river_reaches_in_aoi': 66, 'buffer_polygons': 4, 's2_scene_count': 22, 'feature_table_rows': 1600, 'feature_table_builtup_pct': np.float64(50.0)}
--- Gatharaini ---


{'river_reaches_in_aoi': 21, 'buffer_polygons': 2, 's2_scene_count': 22, 'feature_table_rows': 1600, 'feature_table_builtup_pct': np.float64(50.0)}
--- Motoine ---


{'river_reaches_in_aoi': 41, 'buffer_polygons': 8, 's2_scene_count': 22, 'feature_table_rows': 1600, 'feature_table_builtup_pct': np.float64(50.0)}


,river_reaches_in_aoi,buffer_polygons,s2_scene_count,feature_table_rows,feature_table_builtup_pct
Kasarani,66.0,4.0,22.0,1600.0,50.0
Gatharaini,21.0,2.0,22.0,1600.0,50.0
Motoine,41.0,8.0,22.0,1600.0,50.0
